In [0]:
%sql
WITH m AS (SELECT * FROM laddstolpar_df.gold.mart_kommun_metric),
w AS (
  SELECT a.kommun_kod,
         a.pct_rank    AS g_shrunk,
         r.pct_rank    AS g_raw,
         d.pct_rank    AS demand,
         a.denominator AS n_new_cars,
         a.value       AS d_shrunk,
         r.value       AS d_raw
  FROM m a
  JOIN m r USING (kommun_kod)
  JOIN m d USING (kommun_kod)
  WHERE a.metric = 'growth' AND r.metric = 'growth_raw' AND d.metric = 'ev_demand'
)
SELECT
  ROUND(corr(g_raw, demand), 2)                         AS rho_raw_demand,
  ROUND(corr(g_shrunk, demand), 2)                      AS rho_shrunk_demand,
  ROUND(corr(g_shrunk, g_raw), 2)                       AS rho_shrunk_raw,
  CAST(percentile(n_new_cars, 0.5) AS BIGINT)           AS median_new_cars,
  COUNT_IF(n_new_cars < 46)                             AS kommuner_below_k,
  ROUND(100 * AVG(ABS(d_shrunk - d_raw)), 2)            AS mean_shift_pp,
  ROUND(100 * MAX(ABS(d_shrunk - d_raw)), 2)            AS max_shift_pp
FROM w;